# 1 - Imports & helper functions

## 1.1 - Imports

In [ ]:
# system utilities
import os
from dotenv import load_dotenv
from tqdm.notebook import tqdm
import re
import html
import time

# email processing
from email.header import decode_header
import email
from email import policy
import imaplib

# data processing
import pandas as pd

# s3 backup
import boto3
from botocore.exceptions import NoCredentialsError

# custom email functions
from utils import *


# Maximise Pandas df display area
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.expand_frame_repr', False)

## 1.2 Load environment variables from .env file

In [ ]:
load_dotenv()
IMAP_SERVER = "export.imap.mail.yahoo.com"
EMAIL_ACCOUNT = os.getenv("YAHOO_EMAIL")
APP_PASSWORD = os.getenv("YAHOO_APP_PASSWORD")

print("Credentials loaded successfully!")

## 1.3 - Storage directories

### 1.3.1 - Local

In [ ]:
BASE_DIR = "./yahoo_local_archive"
EML_DIR = os.path.join(BASE_DIR, "emls")
ATTACH_DIR = os.path.join(BASE_DIR, "attachments")

os.makedirs(EML_DIR, exist_ok=True)
os.makedirs(ATTACH_DIR, exist_ok=True)

### 1.3.2 - Remote

In [ ]:
bucket_name = os.getenv('AWS_BUCKET_NAME') 
# s3 client
s3_client = boto3.client("s3")

## 1.4 - Helper functions

In [ ]:
def process_single_folder(mail, folder_name, batch_size=500):
  """Processes a folder in chunks, logging in fresh per batch to bypass

  Yahoo's aggressive long-session throttling and socket drops.
  """
  print(f"\n--- Processing folder: '{folder_name}' ---")

  # First, just get the total IDs while we have the initial connection
  try:
    status, data = mail.select(f'"{folder_name}"', readonly=True)
    if status != "OK":
      print(f"Could not open folder '{folder_name}'. Skipping.")
      return []
  except Exception as e:
    print(f"Error selecting folder '{folder_name}': {e}. Skipping.")
    return []

  status, messages = mail.search(None, "ALL")
  if status != "OK":
    print(f"Error searching messages in '{folder_name}'.")
    return []

  email_ids = messages[0].split()
  total_emails = len(email_ids)
  print(
      f"Found {total_emails} emails in '{folder_name}'. Processing in batches"
      f" of {batch_size} with fresh reconnections..."
  )

  folder_metadata = []
  total_bytes_downloaded = 0
  total_eml_bytes = 0
  total_attach_bytes = 0

  batches = [
      email_ids[i : i + batch_size]
      for i in range(0, total_emails, batch_size)
  ]

  with tqdm(
      total=total_emails, desc=f"Downloading {folder_name}", unit="email"
  ) as pbar:
    for batch_idx, batch in enumerate(batches):

      # --- FRESH RECONNECTION PER BATCH TO AVOID YAHOO THROTTLING ---
      active_mail = mail
      for attempt in range(3):
        try:
          active_mail.noop()  # Test if connection is alive
          break
        except Exception:
          # Reconnect if dropped
          try:
            active_mail = imaplib.IMAP4_SSL(IMAP_SERVER, 993)
            active_mail.login(EMAIL_ACCOUNT, APP_PASSWORD)
            active_mail.select(f'"{folder_name}"', readonly=True)
            break
          except Exception:
            time.sleep(3)

      if batch_idx > 0:
        time.sleep(3)  # Gentle cooldown between batches

      for e_id in batch:
        success = False
        for fetch_attempt in range(3):
          try:
            res, msg_data = active_mail.fetch(e_id, "(RFC822)")
            if res == "OK":
              success = True
              break
          except Exception:
            # If socket drops mid-batch, instantly re-login and retry
            try:
              active_mail = imaplib.IMAP4_SSL(IMAP_SERVER, 993)
              active_mail.login(EMAIL_ACCOUNT, APP_PASSWORD)
              active_mail.select(f'"{folder_name}"', readonly=True)
              res, msg_data = active_mail.fetch(e_id, "(RFC822)")
              if res == "OK":
                success = True
                break
            except Exception:
              time.sleep(2)

        if not success:
          pbar.update(1)
          continue

        try:
          raw_email = msg_data[0][1]
          msg = email.message_from_bytes(raw_email, policy=policy.default)

          uid = e_id.decode("utf-8")
          safe_folder_prefix = "".join(
              c if c.isalnum() else "_" for c in folder_name
          )
          unique_id = f"{safe_folder_prefix}_{uid}"

          # 1. Save raw .eml file locally
          eml_filename = f"email_{unique_id}.eml"
          eml_path = os.path.join(EML_DIR, eml_filename)
          with open(eml_path, "wb") as f:
            f.write(raw_email)

          file_size = os.path.getsize(eml_path)
          total_eml_bytes += file_size
          total_bytes_downloaded += file_size

          # 2. Extract Metadata fields
          subject = decode_str(msg.get("Subject"))
          sender = decode_str(msg.get("From"))
          recipient = decode_str(msg.get("To"))
          cc = decode_str(msg.get("Cc"))
          bcc = decode_str(msg.get("Bcc"))
          date = decode_str(msg.get("Date"))

          body = extract_body(msg)
          snippet = clean_snippet(body, max_len=500)

          # 3. Handle Attachments
          has_attachments = 0
          attachment_names_list = []
          attachment_extensions_set = set()
          email_attach_dir = os.path.join(ATTACH_DIR, f"email_{unique_id}")

          for part in msg.walk():
            if part.get_content_maintype() == "multipart":
              continue
            if part.get("Content-Disposition") is None:
              continue

            filename = part.get_filename()
            if filename:
              has_attachments = 1
              if not os.path.exists(email_attach_dir):
                os.makedirs(email_attach_dir)
              filename = decode_str(filename)
              attachment_names_list.append(filename)

              _, ext = os.path.splitext(filename)
              if ext:
                attachment_extensions_set.add(ext.lower())

              filepath = os.path.join(email_attach_dir, filename)
              with open(filepath, "wb") as f:
                f.write(part.get_payload(decode=True))

              attach_size = os.path.getsize(filepath)
              total_attach_bytes += attach_size
              total_bytes_downloaded += attach_size

          extensions_str = ", ".join(sorted(attachment_extensions_set))

          metadata_record = {
              "folder": folder_name,
              "uid": uid,
              "sender": sender,
              "recipient": recipient,
              "cc": cc,
              "bcc": bcc,
              "date": date,
              "subject": subject,
              "has_attachments": has_attachments,
              "attachment_names": attachment_names_list,
              "attachment_extensions": extensions_str,
              "eml_path": eml_path,
              "body_snippet": snippet,
          }
          folder_metadata.append(metadata_record)

        except Exception as e:
          tqdm.write(
              f"Error parsing message ID {e_id} in '{folder_name}': {e}"
          )

        pbar.update(1)

  # Storage Summary calculation
  eml_mb = total_eml_bytes / (1024 * 1024)
  attach_mb = total_attach_bytes / (1024 * 1024)
  total_mb = total_bytes_downloaded / (1024 * 1024)

  eml_str = f"{eml_mb / 1024:.2f} GB" if eml_mb >= 1024 else f"{eml_mb:.2f} MB"
  attach_str = (
      f"{attach_mb / 1024:.2f} GB" if attach_mb >= 1024 else f"{attach_mb:.2f} MB"
  )
  total_str = (
      f"{total_mb / 1024:.2f} GB" if total_mb >= 1024 else f"{total_mb:.2f} MB"
  )

  print(
      f"📊 Storage Summary for '{folder_name}': Messages: {eml_str} |"
      f" Attachments: {attach_str} | Total: {total_str} downloaded."
  )

  return folder_metadata

In [ ]:
def run_batch_extraction_from_csv(folder, csv_path="yahoo_folders.csv"):
  """Connects to the Yahoo IMAP server and executes the extraction pipeline

  for a single specified folder, cross-referencing against a CSV folder list.

  Parameters:
  ----------
  folder : str
      The specific name of the mail folder to process.
  csv_path : str, optional
      The file path to the CSV containing the list of valid folder names
      (default is "yahoo_folders.csv").

  Returns:
  -------
  pandas.DataFrame
      A DataFrame containing the metadata records of all emails successfully
      extracted and processed from the target folder.

  Notes:
  -----
  - Reads the provided CSV file to validate whether the requested folder
    exists in your master configuration list (issues a warning if unmatched).
  - Establishes a secure SSL IMAP connection using environment credentials.
  - Delegates message fetching and file storage to `process_single_folder()`.
  - Automatically logs out of the IMAP server upon completion.
  """
 

  df = pd.read_csv(csv_path)
  valid_folders = df["folder_name"].dropna().tolist()
  if folder not in valid_folders:
    print(f"Warning: '{folder}' not found in CSV, proceeding anyway...")

  print(f"Connecting to {IMAP_SERVER}...")
  mail = imaplib.IMAP4_SSL(IMAP_SERVER, 993)
  mail.login(EMAIL_ACCOUNT, APP_PASSWORD)

  # 1. Process and download the folder locally
  records = process_single_folder(mail, folder)

  # --- SAFELY HANDLE LOGOUT SOCKET DROPS AT THE END ---
  try:
    mail.logout()
  except Exception:
    print("Note: IMAP socket closed/timed out during logout (harmless, data is safe).")
    
  print(f"\n Finished processing folder: '{folder}' locally!")
  
  return pd.DataFrame(records)

In [ ]:
def verify_downloaded_eml(eml_path, expected_size):
  """Verifies that the saved .eml file exists, matches expected size,

  and can be successfully re-parsed without corruption.
  """
  if not os.path.exists(eml_path):
    return False

  # 1. Check file size
  actual_size = os.path.getsize(eml_path)
  if actual_size != expected_size:
    return False

  # 2. Try parsing it back
  try:
    with open(eml_path, "rb") as f:
      msg = email.message_from_binary_file(f, policy=policy.default)
      # Basic structural sanity check
      if not msg.get("Subject") and not msg.get("From"):
        # Even if headers are sparse, if it parsed without crashing, it's structurally valid
        pass
    return True
  except Exception:
    return False

In [ ]:
def upload_folder_to_s3(folder_name, local_eml_dir=EML_DIR, local_attach_dir=ATTACH_DIR, bucket_name=bucket_name, s3_client=s3_client):
  """Walks through local directories and uploads files belonging to the specified

  folder to a matching sub-folder structure in S3, with tqdm progress bars.
  """
  # s3_client = boto3.client("s3")
  
  # Create a clean safe name for the S3 prefix path
  safe_folder_name = "".join(c if c.isalnum() or c in (' ', '_', '-') else "_" for c in folder_name).strip()
  safe_folder_name = safe_folder_name.replace(" ", "_")
  
  print(f"\n☁️ Uploading local archives for '{folder_name}' to S3 bucket '{bucket_name}/{safe_folder_name}/'...")

  uploaded_count = 0

  # Gather all matching .eml files first so we know the total count for tqdm
  eml_files_to_upload = []
  if os.path.exists(local_eml_dir):
    eml_files_to_upload = [
        f for f in os.listdir(local_eml_dir) 
        if f.startswith(f"email_{safe_folder_name}_")
    ]

  # 1. Progress bar for .eml files
  if eml_files_to_upload:
    for filename in tqdm(eml_files_to_upload, desc=f"Uploading EMLs ({folder_name})", unit="file"):
      local_path = os.path.join(local_eml_dir, filename)
      s3_key = f"{safe_folder_name}/emls/{filename}"
      try:
        s3_client.upload_file(local_path, bucket_name, s3_key)
        uploaded_count += 1
      except Exception as e:
        tqdm.write(f"Error uploading {filename} to S3: {e}")

  # Gather all matching attachment files first
  attachment_files_to_upload = []
  if os.path.exists(local_attach_dir):
    for item_name in os.listdir(local_attach_dir):
      if item_name.startswith(f"email_{safe_folder_name}_"):
        email_attach_subfolder = os.path.join(local_attach_dir, item_name)
        if os.path.isdir(email_attach_subfolder):
          for root, dirs, files in os.walk(email_attach_subfolder):
            for file in files:
              local_path = os.path.join(root, file)
              relative_path = os.path.relpath(local_path, local_attach_dir)
              attachment_files_to_upload.append((local_path, relative_path))

  # 2. Progress bar for attachments (if any exist)
  if attachment_files_to_upload:
    for local_path, relative_path in tqdm(attachment_files_to_upload, desc=f"Uploading Attachments", unit="file"):
      s3_key = f"{safe_folder_name}/attachments/{relative_path}"
      try:
        s3_client.upload_file(local_path, bucket_name, s3_key)
        uploaded_count += 1
      except Exception as e:
        tqdm.write(f"Error uploading attachment to S3: {e}")

  print(f"✅ Successfully uploaded {uploaded_count} items for '{folder_name}' to S3!")

In [ ]:
def audit_yahoo_archives_with_progress(
    csv_path="yahoo_folders.csv", bucket_name=bucket_name
):
  """Audits local disk counts vs S3 counts, handling spaces/underscores in

  prefixes.
  """
  df_folders = pd.read_csv(csv_path)
  folder_list = df_folders["folder_name"].dropna().tolist()

  audit_data = []
  paginator = s3_client.get_paginator("list_objects_v2")

  for folder in tqdm(folder_list, desc="Auditing Yahoo Folders", unit="folder"):
    # Safe prefix used for local disk naming (spaces -> underscores)
    safe_prefix = "".join(c if c.isalnum() else "_" for c in folder)

    # Possible S3 prefixes to check (with spaces or with underscores)
    possible_s3_prefixes = [f"{folder}/", f"{safe_prefix}/"]

    # 1. Count local .eml files
    local_eml_count = 0
    if os.path.exists(EML_DIR):
      local_eml_count = sum(
          1
          for f in os.listdir(EML_DIR)
          if f.startswith(f"email_{safe_prefix}_") and f.endswith(".eml")
      )

    # 2. Count S3 .eml objects by trying the possible prefix variants
    s3_eml_count = 0
    success = False

    for s3_prefix in possible_s3_prefixes:
      try:
        temp_count = 0
        for page in paginator.paginate(Bucket=bucket_name, Prefix=s3_prefix):
          if "Contents" in page:
            for obj in page["Contents"]:
              if obj["Key"].endswith(".eml"):
                temp_count += 1
        if temp_count > 0:
          s3_eml_count = temp_count
          success = True
          break
      except Exception:
        continue

    # If 0 files found on both, it might genuinely be 0 or an empty folder
    if not success and local_eml_count == 0:
      s3_eml_count = 0

    audit_data.append({
        "folder_name": folder,
        "local_eml_count": local_eml_count,
        "s3_eml_count": s3_eml_count,
        "match": local_eml_count == s3_eml_count,
    })

  df_audit = pd.DataFrame(audit_data)
  return df_audit

# 2 - Batch Extraction Across Multiple Folders

## 2.1 - Local backup

In [ ]:
# Option A: Read all folders from your master CSV
df_folders_master = pd.read_csv("yahoo_folders.csv")
folders_to_process = df_folders_master["folder_name"].dropna().tolist()

# Option B: Alternatively, define a custom subset if you only want specific folders
# folders_to_process = ["Inbox", "Sent", "Archive"]

print(f"📂 Target folders queued for local extraction: {folders_to_process}\n")

In [ ]:
# Master list to accumulate metadata across batches if running in a single session
all_extracted_records = []

for folder_name in folders_to_process:
    print(f"\n==========================================")
    print(f"🚀 Starting extraction for: '{folder_name}'")
    print(f"==========================================")
    
    # Run the extraction for the current folder
    df_folder_result = run_batch_extraction_from_csv(folder=folder_name, csv_path="yahoo_folders.csv")
    
    if not df_folder_result.empty:
        all_extracted_records.append(df_folder_result)
        
        # Optional: Save incremental CSV per folder immediately so data is never lost on kernel reset
        safe_name = "".join(c if c.isalnum() else "_" for c in folder_name)
        incremental_csv = f"metadata_{safe_name}.csv"
        df_folder_result.to_csv(incremental_csv, index=False, encoding="utf-8")
        print(f"💾 Incremental metadata saved locally to '{incremental_csv}'")
    else:
        print(f"⚠️ No messages found or extracted for '{folder_name}'.")

# Combine all batch results if any records were collected
if all_extracted_records:
    df_global_metadata = pd.concat(all_extracted_records, ignore_index=True)
    df_global_metadata.to_csv("yahoo_metadata_master_local.csv", index=False, encoding="utf-8")
    print(f"\n✨ Local extraction batch complete! Total records compiled: {len(df_global_metadata)}")
    print("📁 Master metadata saved to 'yahoo_metadata_master_local.csv'")
else:
    print("\n⚠️ No records were extracted across the specified folders.")

## 2.2 - Optional S3 Cloud Backup Loop

In [ ]:
# Run this cell independently only when you are ready to back up your local archives to AWS.

ENABLE_S3_BACKUP = False  # Change to True to execute cloud sync

if ENABLE_S3_BACKUP:
    print("☁️ Initializing optional S3 backup sync...")
    
    # Use the same folder list or read from your CSV
    folders_to_sync = folders_to_process # or define a custom subset
    
    for folder_name in folders_to_sync:
        # Check if local files exist for this folder before attempting upload
        safe_folder_name = "".join(c if c.isalnum() or c in (' ', '_', '-') else "_" for c in folder_name).strip().replace(" ", "_")
        
        eml_exists = any(f.startswith(f"email_{safe_folder_name}_") for f in os.listdir(EML_DIR)) if os.path.exists(EML_DIR) else False
        
        if eml_exists:
            upload_folder_to_s3(folder_name)
        else:
            print(f"⏭️ No local archives found for '{folder_name}' on disk. Skipping S3 upload.")
            
    print("\n🚀 Optional S3 sync process finished!")
else:
    print("ℹ️ S3 backup is currently disabled. Set 'ENABLE_S3_BACKUP = True' when you wish to sync your local files to the cloud.")

# 3 - Audit before deletion

**Some sanity checks to run on your local & optional remote backups before you decide to delete any folders from your Yahoo email.**

In [ ]:
# Check what top-level prefixes/folders actually exist in S3 bucket
response = s3_client.list_objects_v2(
    Bucket=bucket_name, Delimiter='/'
)

print("Top-level folders found in S3 bucket:")
if 'CommonPrefixes' in response:
  for prefix in response['CommonPrefixes']:
    print(f" - {prefix['Prefix']}")
else:
  print("No folder prefixes found at root.")

In [ ]:
# Run the updated audit
df_audit_results = audit_yahoo_archives_with_progress()

# Display mismatches
mismatches = df_audit_results[df_audit_results["match"] == False]
if len(mismatches) == 0:
  print("\n✨ Audit complete! All folders match 100% between local disk and S3.")
else:
  print(f"\n⚠️ Found {len(mismatches)} folders with count mismatches:")
  display(mismatches)